# DEEZER API PULL

11/17/26 script to pull deezer data from their public API.
- have genre
- can pull top 50 artists in each genre (url format: base + /genre + /genre_id + /artists)
- goal: dataframe of top 50 artists per genre

functions:
- get genre
- get artists from genre
- validation
- create dataframe
- upload to s3 in parquet form for efficiency (at least as a best practice)

In [31]:
import requests
import pandas as pd

import io
import time
from datetime import datetime, timezone

import boto3
from botocore.exceptions import ClientError

# variables and session setup

In [32]:
MAX_RETRIES = 3
REQUEST_DELAY = 0.12   # seconds between calls; Deezer allows ~50 per 5s
url = 'https://api.deezer.com'
BUCKET = 'deezer-api-practice'

TRACKS_PER_GENRE = 100
MIN_GENRES = 15
MIN_TRACKS = TRACKS_PER_GENRE * 0.8   # a genre below this many tracks is a problem
MAX_NULL_FRACTION = 0.05

session = requests.Session()

In [33]:
# error class so we know where the error comes from further down the line/when the script is run
class DeezerError(Exception):
    pass

# DEFINE FUNCTIONS

In [34]:
def get_genre_data() -> pd.DataFrame:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            time.sleep(REQUEST_DELAY)
            response = session.get(f'{url}/genre', timeout=15)
            response.raise_for_status()
            body = response.json() # the whole response
            if "error" in body:
                raise DeezerError('error in genres data body')
            df = pd.DataFrame(body["data"])[['id', 'name']]
            return df[df['id'] != 0] # no 'all' genre
        except (requests.RequestException, DeezerError) as e:
            print(f"attempt {attempt} failed:", e)
            if attempt == MAX_RETRIES:
                raise
            time.sleep(2 ** attempt)

genres = get_genre_data()

In [35]:
def get_top_tracks() -> pd.DataFrame:
    result = []
    for genre_id, genre in zip(genres['id'], genres['name']):
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                time.sleep(REQUEST_DELAY)
                response = session.get(
                    f'{url}/chart/{genre_id}/tracks',
                    params={'limit': TRACKS_PER_GENRE},
                    timeout=15)
                response.raise_for_status()
                body = response.json() # the whole response
                if "error" in body:
                    raise DeezerError('error in tracks data body')
                break
            except (requests.RequestException, DeezerError) as e:
                print(f"genre {genre_id}, attempt {attempt} failed:", e)
                if attempt == MAX_RETRIES:
                    raise
                time.sleep(2 ** attempt)
        df = pd.DataFrame(body['data'])
        df['Genre'] = genre # add in genre data
        result.append(df)
    return pd.concat(result, ignore_index=True)

top_tracks = get_top_tracks()

In [36]:
# expanding out the artist column from a dictionary to separate columns to be added back to the tracks dataframe
artist_info_df = pd.json_normalize(top_tracks['artist'])
artist_info_df = artist_info_df.rename(columns={'id':'artist_id',
                                                'name':'artist_name',
                                                'link':'artist_link',
                                                'picture':'artist_picture',
                                                'tracklist':'top_tracklist'}, errors='ignore')[['artist_id','artist_name','artist_link','artist_picture','top_tracklist']]

In [37]:
# expanding out the album column from a dictionary to separate columns to be added back to the tracks dataframe
album_info_df = pd.json_normalize(top_tracks['album'])
album_info_df = album_info_df.rename(columns={'id':'album_id',
                                              'title':'album_title',
                                              'cover':'album_cover',
                                              'tracklist':'album_tracklist'}, errors='ignore')[['album_id','album_title','album_cover','album_tracklist']]

In [38]:
top_tracks_expanded = pd.concat([top_tracks, artist_info_df, album_info_df], axis=1).drop(columns=['artist', 'album'], errors='ignore')

In [39]:
# deduplicated artist list
artist_list = top_tracks_expanded['artist_id'].unique().tolist()

In [ ]:
# def get_artist_data() -> pd.DataFrame:
#     result = []
#     for artist_id in artist_list:
#         for attempt in range(1, MAX_RETRIES + 1):
#             try:
#                 time.sleep(REQUEST_DELAY)
#                 response = session.get(f'{url}/artist/{artist_id}', timeout=15)
#                 response.raise_for_status()
#                 body = response.json() # the whole response
#                 if "error" in body:
#                     raise DeezerError('error in artists data body')
#                 break
#             except (requests.RequestException, DeezerError) as e:
#                 print(f"artist {artist_id}, attempt {attempt} failed:", e)
#                 if attempt == MAX_RETRIES:
#                     raise
#                 time.sleep(2 ** attempt)
#         result.append(body)
#     return pd.DataFrame(result)

# artists_data = get_artist_data()

using parallel workers to stay safely under Lambda's 15 minute limit (non-parallel was taking around 10 mins)

In [44]:
import threading
from concurrent.futures import ThreadPoolExecutor

MAX_WORKERS = 5

_rate_lock = threading.Lock()
_next_call = 0.0

def wait_for_slot():
    """Shared across threads: hands out call times REQUEST_DELAY apart."""
    global _next_call
    with _rate_lock:
        now = time.monotonic()
        start = max(now, _next_call)
        _next_call = start + REQUEST_DELAY
    if start > now:
        time.sleep(start - now)


def fetch_artist(artist_id) -> dict:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            wait_for_slot()
            response = session.get(f'{url}/artist/{artist_id}', timeout=15)
            response.raise_for_status()
            body = response.json() # the whole response
            if "error" in body:
                raise DeezerError('error in artists data body')
            return body
        except (requests.RequestException, DeezerError) as e:
            print(f"artist {artist_id}, attempt {attempt} failed:", e)
            if attempt == MAX_RETRIES:
                raise
            time.sleep(2 ** attempt)


def get_artist_data():
    pool = ThreadPoolExecutor(max_workers=MAX_WORKERS)
    try:
        result = list(pool.map(fetch_artist, artist_list))   # keeps the artist_list order
    except Exception:
        pool.shutdown(wait=False, cancel_futures=True)   # stop queued calls on failure
        raise
    pool.shutdown()
    return pd.DataFrame(result)

artists_data = get_artist_data()

# DATA VALIDATION

add checks for nulls, missing genres, artist names, etc

In [41]:
# Raised when fetched data fails a check
class DataQualityError(Exception):
    pass

def validate_genres(df = genres):
    problems = []
    if len(df) < MIN_GENRES:
        problems.append(f"Error: genres has {len(df)} rows, expected at least {MIN_GENRES}")
    if df.isna().any().any():
        problems.append("Error: null values in genres df")
    return problems

def validate_tracks(df = top_tracks_expanded):
    problems = []
    per_genre = df.groupby('Genre').size()
    short = per_genre[per_genre < MIN_TRACKS]
    if len(short):
        problems.append(f"Error: top_tracks: under {MIN_TRACKS:.0f} tracks: {short.to_dict()}")
    nulls = df[['id', 'title', 'artist_id', 'artist_name']].isna().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        problems.append(f"top_tracks: nulls in {nulls.to_dict()}")
    return problems

def validate_artists(df = artists_data):
    problems = []
    if df.empty:
        return ["artists: no rows"]
    if df[['id', 'name']].isna().any().any():
        problems.append("artists: null id or name")
    if df['nb_fan'].isna().mean() > MAX_NULL_FRACTION:
        problems.append("artists: too many null nb_fan values")
    if df['id'].duplicated().any():
        problems.append("artists: duplicate ids")
    return problems

def validate_all() -> None:
    problems = (validate_genres()
                + validate_tracks()
                + validate_artists())
    if problems:
        raise DataQualityError("\n".join(problems))
    print("Validation passed")


# UPLOAD TO S3

In [42]:
s3 = boto3.client('s3')

#Write a DataFrame to S3 as Parquet
def upload_df(df: pd.DataFrame, bucket: str, key: str) -> None:
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False)   # needs pyarrow
    buffer.seek(0)
    s3.upload_fileobj(buffer, bucket, key)

In [43]:
validate_all()

now = datetime.now(timezone.utc)
load_date = now.strftime('%Y-%m-%d')  # computed once so all files share a date

# one dated folder per dataset: <dataset>/dt=<date>/<dataset>.parquet
s3_datasets = {'genres': genres,
    'top_tracks': top_tracks_expanded,
    'artists': artists_data}

for name, df in s3_datasets.items():
    df = df.copy()                 # don't change the original DataFrames
    df['load_date'] = load_date    # which day's run this row came from
    df['fetched_at'] = now         # exact UTC time of the run
    upload_df(df, BUCKET, f'{name}/dt={load_date}/{name}.parquet')   # raises on failure
    print(f'uploaded {name}')

Validation passed
uploaded genres
uploaded top_tracks
uploaded artists


# add all this to main so that it's a python script and package
## Before Lambda automation (checklist)
- Turn the notebook into `deezer_pull.py` with a `main()` and a thin `lambda_handler(event, context)`

**AWS**
- Lambda: Python runtime, timeout up to 900 s, memory ~512 MB+ (pandas), log group with retention.
- `requests` is not in the Lambda runtime: package it. For pandas + pyarrow use the AWS SDK for pandas managed layer (check the ARN for your Region), or write JSON Lines instead to avoid the heavy dependencies.
- No credentials in code: the function's execution role needs only `s3:PutObject` on `deezer-api-practice/*` plus logging.
- Schedule with EventBridge Scheduler (daily, UTC); allow a `load_date` in the event for manual re-runs (same-day re-runs overwrite that partition).
- Alarm on Lambda errors (CloudWatch alarm -> SNS email) so a failed day does not go unnoticed.